# Cell-Type Resolution: HuBMAP


## What does HuBMAP add?

GTEx showed that genes from the heart-failure paper are expressed in bulk heart
tissue. We now ask which sampled heart cell types contain their RNA.

HuBMAP stands for **Human BioMolecular Atlas Program**. This NIH Common Fund
program maps cells and molecules within human tissues. Its Cells API can connect
gene expression to cell labels such as ventricular cardiac myocyte,
fibroblast, or macrophage.

Cell-level results depend on the samples, cell labels, processing, and group
size. HuBMAP adds cell-type context to the paper's genes. It does not test the
effects of the 54 individual variants.

## How the HuBMAP API Works

The Cells API first creates query handles for the heart and ventricular cardiac
myocytes. It intersects those sets, samples up to 500 cells, and requests one
gene at a time. Missing indexed values are recorded as unavailable, not zero.

First, load the published variants and the shared API helper.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from api_helpers import fetch_hubmap_ventricular_context

DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")


Next, query ventricular cardiac myocytes for the 25 unique genes. This request
can take longer than the other API calls. The commented line loads all six
cell types from the saved response if the live service is unavailable.


In [ ]:
gene_symbols = sorted(variants["gene_symbol"].unique())
hubmap = fetch_hubmap_ventricular_context(gene_symbols)

# Backup: use the frozen 2026-08-11 response instead of the live API.
# hubmap = pd.read_csv(DATA_DIR / "hubmap_cell_expression.csv")

hubmap.head()


### Live data and backup
The default code queries HuBMAP. Live coverage can change. If the request
fails, comment out the live line and uncomment the saved-data line.

## Review Ventricular Cardiac Myocytes

Return to the paper's 25 genes and examine one disease-relevant cell type.

First, keep the ventricular cardiac-myocyte rows and count the genes with and
without indexed expression values.


In [ ]:
ventricular = hubmap[
    hubmap["cell_type_id"] == "CL:0002131"
].copy()
availability_summary = (
    ventricular["availability"]
    .value_counts()
    .rename_axis("availability")
    .reset_index(name="genes")
)
availability_summary


Next, display the ten available genes with the highest mean expression in the
500-cell teaching sample.


In [ ]:
top_ventricular_genes = (
    ventricular[ventricular["availability"] == "available"]
    .nlargest(10, "mean_normalized_expression")
    .set_index("gene_symbol")
)
top_ventricular_genes.loc[
    :, ["mean_normalized_expression", "percent_detected"]
]


Finally, plot those mean expression values. Interpret the chart with the
percent-detected values and missing-gene count above.


In [ ]:
axis = top_ventricular_genes["mean_normalized_expression"].plot.bar(
    color="#3d64b3",
    figsize=(10, 5),
)
axis.set_ylabel("Mean normalized expression")
axis.set_xlabel("Gene from the source paper")
axis.set_title("HuBMAP ventricular cardiac-myocyte context")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
